In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [3]:
train_df = pd.read_csv("../data/processed/train_final.csv")
test_df = pd.read_csv("../data/processed/test_final.csv")

print(train_df.shape)
print(test_df.shape)

(1486, 7)
(372, 7)


In [4]:
features = [
    "log_pl_orbsmax",
    "log_st_lum",
    "log_pl_rade",
    "log_teq",
    "log_st_teff",
    "log_stellar_flux"
]

X_train = train_df[features]
y_train = train_df["habitable"]

X_test = test_df[features]
y_test = test_df["habitable"]

In [5]:
print(y_train.value_counts())

habitable
0    1478
1       8
Name: count, dtype: int64


In [6]:
train_df.shape

(1486, 7)

In [7]:
test_df.shape

(372, 7)

In [8]:
print(test_df["habitable"].value_counts())

habitable
0    370
1      2
Name: count, dtype: int64


In [9]:
print(train_df.head())
print(test_df.head())

   log_pl_orbsmax  log_st_lum  log_pl_rade   log_teq  log_st_teff  \
0       -0.255128    0.903445    -0.360182  0.531316     0.515772   
1       -0.257927    0.695227    -0.767733  0.476395     0.481507   
2       -0.122958   -0.321166    -0.391518 -0.366665     0.161508   
3       -0.341789    0.298404     1.398775  0.848777     0.871896   
4        3.458908    0.392995     1.099040 -2.636207     0.696510   

   log_stellar_flux  habitable  
0          0.595122          0  
1          0.515846          0  
2         -0.677869          0  
3          1.054154          0  
4         -1.885888          0  
   log_pl_orbsmax  log_st_lum  log_pl_rade   log_teq  log_st_teff  \
0       -0.295167   -0.841653    -1.318503 -0.248592    -0.896215   
1       -0.206028   -0.822152    -0.623542 -0.588329    -1.049192   
2        5.161499   -1.019647     1.196129 -5.353629    -3.294966   
3       -0.332822    0.544322     1.039218  0.859548     1.200465   
4       -0.204441   -0.022023    -1.357912

In [10]:
features = [
    "log_pl_orbsmax",
    "log_st_lum",
    "log_pl_rade",
    "log_teq",
    "log_st_teff",
    "log_stellar_flux"
]

X_train = train_df[features]
y_train = train_df["habitable"]

X_test = test_df[features]
y_test = test_df["habitable"]

print(X_train.shape)
print(X_test.shape)

(1486, 6)
(372, 6)


In [11]:
pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

In [13]:
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="recall",
    cv=5,
    n_jobs=-1
)

In [14]:
grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Recall:")
print(grid_search.best_score_)

Best Parameters:
{'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Best Cross Validation Recall:
1.0


c:\Users\Samyukthaa\exoplanet-habitability-project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [15]:
best_model = grid_search.best_estimator_

print(best_model)

Pipeline(steps=[('smote', SMOTE(random_state=42)), ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=0.01, max_iter=1000, penalty='l2'))])


In [16]:
from sklearn.metrics import classification_report, confusion_matrix

# predictions
y_pred = best_model.predict(X_test)

# evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[340  30]
 [  0   2]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       370
           1       0.06      1.00      0.12         2

    accuracy                           0.92       372
   macro avg       0.53      0.96      0.54       372
weighted avg       0.99      0.92      0.95       372

